# Headline numbers + corpus audit

Stage 5 of the 6-stage producer chain — the final stage. Loads the YAML and parquet written by stage 04 and prints two pieces:

1. **Canonical `HEADLINE` sheet** — `prompt_analysis.headline_numbers(data, alt_df=…, parquet=…)`. Same call signature consumer notebooks use.
2. **Audit table** — the human-readable summary of corpus statistics (Files / Sentences / Word tokens / ccVersions / Rule sentences / etc.). Each row references the canonical YAML key its value comes from.

This is the only producer stage that appears in the published Quarto site (it's the most reader-friendly view); stages 00–04 run in the kernel but are hidden from the navbar.

## 12. Canonical HEADLINE sheet

Re-uses `prompt_analysis.headline_numbers()` — the same function consumer notebooks call. Pass `alt_df` (for composite-directiveness range and per-version `mood_marker_pct` extremes) and the per-sentence parquet (for parquet-level threat / causal / rule counts) so the producer's audit covers the full HEADLINE contract.

In [1]:
"""Compute and display the canonical HEADLINE dict.

Prints as YAML so the values are visible in this notebook (saving a copy in the
cell output) without requiring another tool.
"""
import os, sys, pathlib, importlib
sys.path.insert(0, ".")
import pandas as pd
import yaml as _yaml

_here = pathlib.Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [_here, *_here.parents] if (p / "prompt_pipeline.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        f"Could not find prompt_pipeline.py walking up from {_here}. "
        "Run from inside the claude-prompts-analysis repo."
    )
if pathlib.Path.cwd() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

import prompt_analysis
importlib.reload(prompt_analysis)  # pick up edits without restarting the kernel
from prompt_analysis import (
    load_yaml, build_alt_df, headline_numbers, qualitative_phrases, bind_inline_vars,
)

data    = load_yaml()
alt_df  = build_alt_df(data)
parquet = pd.read_parquet("sentences_classified.parquet")

HEADLINE = headline_numbers(data, alt_df=alt_df, parquet=parquet)
PHRASES  = qualitative_phrases(HEADLINE, alt_df=alt_df, parquet=parquet)

# Make every formatted figure available as a plain-name variable for inline {python} expressions in the audit-table cell below.
globals().update(bind_inline_vars(HEADLINE, PHRASES))

print(_yaml.safe_dump(HEADLINE, sort_keys=False, default_flow_style=False))

n_files: 290
n_sentences: 5881
n_word_tokens: 133611
n_versions: 58
n_rule_sentences: 2288
pct_explained_same: 6.6871
pct_explained_para: 24.3444
judgment_count: 78
procedural_count: 595
judgment_to_procedural_ratio: 0.131
threat_count: 8
causal_count: 137
threat_share: 0.0552
question_count: 87
apology_count: 3
selfref_claude: 521
selfref_assistant: 20
selfref_model: 266
pct_anthropomorphic: 0.6456
positive_evaluative_quality: 298
positive_evaluative_emphasis: 185
positive_evaluative_union: 483
negative_evaluative: 152
ratio_quality_to_negative: 1.9605263157894737
ratio_union_to_negative: 3.1776315789473686
appreciative_sent: 4
collaborative_sent: 30
streak_max: 12
n_streaks_ge3: 230
n_streaks_ge5: 52
vocab_hard_prohibitions: 631
vocab_hard_prescriptions: 358
vocab_pronouns_2p: 1397
vocab_pronouns_1p: 185
vocab_profanity: 0
modality_deontic: 261
modality_epistemic: 322
modality_dynamic: 559
mood_marker_pct: 0.7679
top_caps_imperative:
- - IMPORTANT
  - 36
- - NEVER
  - 26
- - MUST
  -

## Audit table

Live corpus statistics — these are the canonical values for every prose mention across the notebooks; any number that disagrees gets reconciled to these. Every figure below is computed live from the YAML.

| Quantity | Value |
|---|---:|
| Files | **`{python} n_files`** |
| Sentences | **`{python} n_sents`** |
| Word tokens | **`{python} f"{HEADLINE['n_word_tokens']:,}"`** |
| ccVersions (distinct) | **`{python} n_versions`** (latest `{python} HEADLINE['judgment_to_procedural_ratio_latest_version_id']`) |
| Rule sentences | **`{python} HEADLINE['n_rule_sentences']`** |
| `pct_explained_same` | **`{python} f"{HEADLINE['pct_explained_same']:.2f}%"`** |
| `pct_explained_para` (per rule sentence) | **`{python} rule_exp_pct`** (the headline rule-explanation rate) |
| Rule-bearing paragraphs (total / explained / unexplained) | **`{python} n_para_rules` / `{python} n_para_rules_explained` / `{python} n_para_rules_unexplained`** |
| `pct_paragraphs_with_rules_explained` / `_unexplained` | **`{python} para_yes_pct` / `{python} para_no_pct`** |
| Avg rule sentences per paragraph (explained / unexplained) | **`{python} rules_per_explained_para` / `{python} rules_per_unexplained_para`** — explains the gap between the per-sentence rate (`{python} rule_exp_pct`) and the per-paragraph rate (`{python} para_yes_pct`) |
| `judgment_count` / `procedural_count` / ratio | **`{python} HEADLINE['judgment_count']` / `{python} HEADLINE['procedural_count']` / `{python} ratio_jp`** |
| `threat_count` / `causal_count` / `threat_share` | **`{python} threat_count` / `{python} causal_count` / `{python} f"{HEADLINE['threat_share']:.4f}"`** (`{python} threat_share` in narrative) |
| `question_count` / `apology_count` | `{python} HEADLINE['question_count']` / `{python} apol_count` |
| `selfref_claude` / `_assistant` / `_model` | `{python} HEADLINE['selfref_claude']` / `{python} HEADLINE['selfref_assistant']` / `{python} HEADLINE['selfref_model']` |
| `pct_anthropomorphic` | **`{python} f"{HEADLINE['pct_anthropomorphic']:.4f}"`** (`{python} selfref_pct`) |
| Imperative streaks: `streak_max` / `n_ge3` / `n_ge5` | `{python} streak_max` / `{python} HEADLINE['n_streaks_ge3']` / `{python} HEADLINE['n_streaks_ge5']` |
| RULES-section paragraphs (in / out, % explained) | `{python} HEADLINE['rules_section_in_paragraphs']` (`{python} f"{HEADLINE['rules_section_in_pct_explained']:.2f}%"`) / `{python} HEADLINE['rules_section_out_paragraphs']` (`{python} f"{HEADLINE['rules_section_out_pct_explained']:.2f}%"`) |
| Modality (deontic / epistemic / dynamic) | `{python} HEADLINE['modality_deontic']` / `{python} HEADLINE['modality_epistemic']` / `{python} HEADLINE['modality_dynamic']` |
| `vocab.hard_prohibitions.count` | **`{python} HEADLINE['vocab_hard_prohibitions']`** |
| `vocab.hard_prescriptions.count` | `{python} HEADLINE['vocab_hard_prescriptions']` |
| `vocab.pronouns_2p.count` | **`{python} HEADLINE['vocab_pronouns_2p']`** |
| `vocab.pronouns_1p.count` | **`{python} HEADLINE['vocab_pronouns_1p']`** |
| `vocab.profanity.count` | `{python} HEADLINE['vocab_profanity']` |
| Stance: positive_evaluative_quality / _emphasis / negative_evaluative | **`{python} HEADLINE['positive_evaluative_quality']` / `{python} HEADLINE['positive_evaluative_emphasis']` / `{python} HEADLINE['negative_evaluative']`** |
| Quality-only positive-vs-negative ratio | **`{python} posneg_ratio`×** (`{python} HEADLINE['positive_evaluative_quality']` / `{python} HEADLINE['negative_evaluative']`) |
| Union positive-vs-negative ratio | **`{python} f"{HEADLINE['ratio_union_to_negative']:.2f}"`×** (`{python} HEADLINE['positive_evaluative_union']` / `{python} HEADLINE['negative_evaluative']`) |
| `appreciative_sent_count` | **`{python} appr_count`** |
| `collaborative_sent_count` | **`{python} HEADLINE['collaborative_sent']`** |